In [0]:
from datetime import datetime, timedelta

from src.scripts.extract_data import ExtractData

In [0]:
def get_month_range(year, month):
    today = datetime.today()
    if today.year == year and today.month == month:
        date_start = datetime(year, month, 1)
        date_end = today
    else:
        date_start = datetime(year, month, 1)
        if month == 12:
            next_month = datetime(year + 1, 1, 1)
        else:
            next_month = datetime(year, month + 1, 1)
        date_end = next_month - timedelta(days=1)
    return date_start, date_end

In [0]:
dbutils.widgets.text("year", "")
dbutils.widgets.text("month", "")
dbutils.widgets.text("endpoint", "")

year_param = dbutils.widgets.get("year")
month_param = dbutils.widgets.get("month")
endpoint = dbutils.widgets.get("endpoint")
# endpoint = "cars"

print(f"Parámetro year recibido: {year_param}")
print(f"Parámetro month recibido: {month_param}")
print(f"Parámetro endpoint recibido: {endpoint}")

current_date = datetime.now()
current_year = current_date.year
current_month = current_date.month

year = year_param if year_param else current_year
month = month_param if month_param else current_month
month_str = f"0{month}" if int(month) < 10 else str(month)
date_start, date_end = get_month_range(int(year), int(month))
# date_start, date_end = get_month_range(2025, 3)
period = f"{year}_{month_str}"
print({
    "date_start": date_start,
    "date_end": date_end,
    "period": period
})

In [0]:
extract_data = ExtractData()

In [0]:
formula_1 = extract_data.extract_meetings(date_start, date_end)

In [0]:
formula_1

In [0]:
formula_1 = extract_data.extract_sessions()

In [0]:
formula_1.head()

In [0]:
formula_1 = extract_data.extract_drivers() 

In [0]:
formula_1.head()

In [0]:
if endpoint == "laps":
    laps = extract_data.extract_laps()
    laps_formula_1_delta_table = spark.createDataFrame(laps)
    laps_formula_1_delta_table.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"formula_1.laps_{period}")
elif endpoint == "cars":
    cars = extract_data.extract_cars()
    cars_formula_1_delta_table = spark.createDataFrame(cars)
    cars_formula_1_delta_table.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"formula_1.cars_{period}")
else:
    print("Invalid endpoint provided. Please choose either 'laps' or 'cars'.")